# Model traning logs

In [ ]:
import numpy as np
from ultralytics import YOLO
from ultralytics.utils.metrics import DetMetrics
import mlflow
from ultralytics import settings

# Update a setting
settings.update({"mlflow": True})

## Configure mlFlow


In [ ]:
import mlflow

# 1. Configure MLflow (Run this once at the top of your notebook)
# This sets the destination for ALL subsequent training runs.
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("inspire-auto-render")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1765292283892, experiment_id='1', last_update_time=1765292283892, lifecycle_stage='active', name='inspire-auto-render', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [ ]:
modelv12 = YOLO("yolo12n.pt")

In [ ]:
def custom_fitness_v12(self):
    """Custom fitness weights: 80% detection, 20% precision."""
    # [P, R, mAP@0.5, mAP@0.5:0.95]
    w = [0.2, 0.2, 0.2, 0.4]  
    return (np.nan_to_num(np.array(self.mean_results())) * w).sum()

# 2. Apply the patch using property()
DetMetrics.fitness = property(custom_fitness_v12)

In [ ]:
# change the fitness function to 0.2 0.2 0.4 0.4
# 
params = {
    "data": "YOLO/data.yaml",
    "task": "detect",
    "mode": "train",
    "epochs": 1,
    "batch": 16,
    "imgsz": 640,
    "patience": 200,
    "hsv_h": 0.1,          # Hue: Full 360 rotation
    "hsv_s": 0.7,          # Saturation: Full range
    "hsv_v": 0.4,          # Value: Full range (Black to White)
    "degrees": 45,        # Rotation: CURRENTLY OFF (See suggestions below)
    "mosaic": 1.0,
    "mixup": 0.2,
    "copy_paste": 0.3,
    "erasing": 0.6,        # Random erasing
    "name": "clean-new-validation-color-with-blur",
}
results = modelv12.train(
    **params
    # data="YOLO/data.yaml",
    # task="detect",
    # mode="train",
    # epochs=250,
    # batch=16,
    # imgsz=640,
    # patience=200,
    # hsv_h=0.1,          # Hue: Full 360 rotation
    # hsv_s=0.7,          # Saturation: Full range
    # hsv_v=0.4,          # Value: Full range (Black to White)
    # degrees=45,        # Rotation: CURRENTLY OFF (See suggestions below)
    # mosaic=1.0,
    # mixup=0.2,
    # copy_paste=0.3,
    # erasing=0.6,        # Random erasing
    # name="clean-new-validation-color-with-blur",
)

New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.221 🚀 Python-3.12.11 torch-2.9.0 CPU (Apple M3 Pro)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=YOLO/data.yaml, degrees=45, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.6, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.1, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=clean-new-validation-color-with-blur4, nbs=64, nms=False, opset=None, 

2025/12/09 16:15:00 INFO mlflow.tracking.fluent: Autologging successfully enabled for keras.
2025/12/09 16:15:00 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/12/09 16:15:00 INFO mlflow.tracking.fluent: Autologging successfully enabled for tensorflow.


MLflow: logging run_id(7012da292e284249aa1ee65cbd6eda6e) to http://127.0.0.1:5000
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /Users/caiofeuser/Developer/inspire/auto_render/runs/detect/clean-new-validation-color-with-blur4
Starting training for 1 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        1/1         0G      2.798      5.206       2.94          5        640: 100% ━━━━━━━━━━━━ 2/2 0.2it/s 9.7s5.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 0.9it/s 1.1s
                   all         10          9          0          0          0          0

1 epochs completed in 0.003 hours.
Optimizer stripped from /Users/caiofeuser/Developer/inspire/auto_render/runs/detect/clean-new-validation-color-with-blur4/weights/last.pt, 5.5MB
Optimizer stripped from /Users/caiofeuser/Developer/inspire/auto_re

In [ ]:
# 4. Export & Log Artifact to the JUST-FINISHED run
# We can get the ID of the run Ultralytics just created
last_run_id = mlflow.last_active_run().info.run_id

with mlflow.start_run(run_id=last_run_id):
    # Export to TFLite
    tflite_path = modelv12.export(format="tflite")
    
    # Upload it to the existing run
    mlflow.log_artifact(tflite_path)
    
    print(f"TFLite model uploaded to run: {last_run_id}")

Ultralytics 8.3.221 🚀 Python-3.12.11 torch-2.9.0 CPU (Apple M3 Pro)
YOLOv12n summary (fused): 159 layers, 2,557,508 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/Users/caiofeuser/Developer/inspire/auto_render/runs/detect/clean-new-validation-color-with-blur4/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (5.2 MB)
requirements: Ultralytics requirements ['tf_keras<=2.19.0', 'sng4onnx>=1.0.1', 'ai-edge-litert>=1.2.0,<1.4.0'] not found, attempting AutoUpdate...
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.7 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/5.1 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/252.7 MB ? eta -:--:--
   ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


ONNX: slimming with onnxslim 0.1.78...
ONNX: export success ✅ 1.6s, saved as '/Users/caiofeuser/Developer/inspire/auto_render/runs/detect/clean-new-validation-color-with-blur4/weights/best.onnx' (10.2 MB)
ERROR ❌ TensorFlow SavedModel: export failure 38.5s: module 'onnx.helper' has no attribute 'float32_to_bfloat16'
🏃 View run clean-new-validation-color-with-blur4 at: http://127.0.0.1:5000/#/experiments/2/runs/7012da292e284249aa1ee65cbd6eda6e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


AttributeError: module 'onnx.helper' has no attribute 'float32_to_bfloat16'